# conv-output-shape — ex2: invert the conv formula to compute SAME-style padding

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-output-shape`. Running the final beacon cell reports progress against the `CNN: Conv output shape` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Conv output shape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-output-shape`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-output-shape"
DD_SUBTOPIC = "CNN: Conv output shape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv output shape — quick refresher

The standard formula (dilation=1):
```
L_out = (L_in + 2*P - K) // S + 1
```
**Inverting for 'SAME' padding.** Sometimes you want `L_out == L_in` (or `ceil(L_in / S)`). Solve the formula for `P`:
```
L_in + 2*P - K = (L_out - 1) * S
P = ((L_out - 1) * S - L_in + K) / 2
```
With `S=1` this collapses to `P = (K - 1) // 2` for odd `K` — the classic 'half-kernel' padding everyone memorises. For `S > 1` or even `K`, the inversion isn't always an integer; you round up (top/right) and floor (bottom/left) to split the asymmetry. ARENA's `Conv2d` does NOT do this for you — the learner must compute the padding themselves.

### Exercise 2 — invert the conv formula to compute SAME-style padding

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the conv output-shape formula in reverse: given desired `L_out`, kernel `K`, and stride `S`, compute the (symmetric) padding `P` needed so that a `Conv1d` produces exactly `L_out`.
> Keywords: conv, same-padding, inverse-formula, shape
> ```

**KCs targeted:** `conv-output-shape-formula`, `conv-shape-batch-pass-through`

Implement `ex2_same_padding(L_in, L_out, K, S)`. Solve the forward formula
```
L_out = (L_in + 2*P - K) // S + 1
```
for `P`, returning the smallest non-negative integer that achieves *at least* `L_out`. Specifically:

1. Algebra: drop the floor first → `P_real = ((L_out - 1) * S - L_in + K) / 2`.
2. Round UP: `P = max(0, ceil(P_real))`.
3. Verify by re-running the forward formula with that `P` — the actual output length must be `>= L_out` and within `S` of it. Return `P`.

Then write a quick sanity check inside your function: assert `(L_in + 2*P - K) // S + 1 >= L_out`. The test will probe the same identity from the outside.

Inputs (all positive ints): `L_in, L_out, K, S`.
Output: integer padding `P >= 0`.

In [ ]:
def ex2_same_padding(L_in, L_out, K, S):
    import math
    P_real = ((L_out - 1) * S - L_in + K) / 2.0
    P = max(0, math.ceil(P_real))
    # Sanity check: forward formula must achieve >= L_out.
    assert (L_in + 2 * P - K) // S + 1 >= L_out
    return P


<details><summary>Solution</summary>

```python
def ex2_same_padding(L_in, L_out, K, S):
    import math
    P_real = ((L_out - 1) * S - L_in + K) / 2.0
    P = max(0, math.ceil(P_real))
    # Sanity check: forward formula must achieve >= L_out.
    assert (L_in + 2 * P - K) // S + 1 >= L_out
    return P
```

**Algebra in one line.** Drop the floor (cast to real), then round up to the nearest integer. The floor in the FORWARD formula means the forward output rounds DOWN, so to guarantee we reach `L_out` we must round the inverse UP.

**Why `ceil` and not `round`.** Banker's rounding (which Python's `round` does at `.5`) would sometimes give `P` that's ONE too small — the forward formula then under-shoots `L_out` by exactly 1. `ceil` is the safe direction.

**ARENA practical use.** When you build a CNN-from-scratch downsampler that halves the spatial extent at every stage, you call `ex2_same_padding(L_in=in_size, L_out=in_size//2, K=kernel, S=2)` at each layer. Without this inverse you'd guess-and-check the padding manually.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()